# Reading in a short story as text sample into Python.

### Step 1: Creating Tokens

In [1]:
%config Completer.use_jedi = False

In [2]:
# Reading the file
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99]) # The print command prints the total number of characters followed by the first 100 characters of this file for illustration purposes.

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### Our goal is to tokenize this 20,479-character short story into individual words and special characters that we can then turn into embeddings for LLM training

### Note that it's common to process millions of articles and hundreds of thousands of
### books -- many gigabytes of text -- when working with LLMs. However, for educational
### purposes, it's sufficient to work with smaller text samples like a single book to
### illustrate the main ideas behind the text processing steps and to make it possible to
### run it in reasonable time on consumer hardware.

### How can we best split this text to obtain a list of tokens? For this, we go on a small
### excursion and use Python's regular expression library re for illustration purposes.

In [3]:
import re # re means regular expression

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

# What re.split(r'(\s)', text) does
# \s → matches any whitespace (space, tab, newline).
# () → capturing group.
# When used in re.split, the delimiter is also returned in the result list.
# So the string is split at every whitespace, but the whitespace itself is kept in the output.

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
# The result is a list of individual words, whitespaces, and punctuation characters.

In [5]:
# Let's modify the regular expression splits on whitespaces (\s) and commas, and periods ([,.]):

In [6]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [7]:
# We can see that the words and punctuation characters are now separate list entries just as we wanted.
# A small remaining issue is that the list still includes whitespace characters. Optionally, we can remove these redundant characters,

In [8]:
result = [item for item in result if item.strip()] # Loop and return the item only if item.strip() returns true. So whitwspaces won't be returned.

print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [9]:
# REMOVING WHITESPACES OR NOT

# When developing a simple tokenizer, whether we should encode whitespaces as separate characters or just remove them depends on our application 
# and its requirements. Removing whitespaces reduces the memory and computing requirements. However, keeping whitespaces can be useful if we train 
# models that are sensitive to the exact structure of the text (for example, Python code, which is sensitive to indentation and spacing). Here, we 
# remove whitespaces for simplicity and brevity of the tokenized outputs. Later, we will switch to a tokenization scheme that includes whitespaces.

In [10]:
# The tokenization scheme we devised above works well on the simple sample text. Let's modify it a bit further so that it can also handle other types 
# of punctuation, such as question marks, quotation marks, and the double-dashes we have seen earlier in the first 100 characters of Edith Wharton's 
# short story, along with additional special characters:

In [11]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [12]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [13]:
print(len(preprocessed))

4690


In [14]:
# Creating Token IDs

# In the previous section, we tokenized Edith Wharton's short story and assigned it to a Python variable called preprocessed. Let's now create 
# a list of all unique tokens and sort them alphabetically to determine the vocabulary size.

# Vocabulary is just a list of tokens which is sorted in alphabetical manner and then each unique token is mapped to a unique integer which is called as token id.

In [15]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [16]:
# After determining that the vocabulary size is 1,130 via the above code, we create the vocabulary and print its first 51 entries for illustration purposes.

In [17]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [18]:
for i,item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [19]:
# As we can see, based on the output above, the dictionary contains individual tokens associated with unique integer labels.

# Later in this book, when we want to convert the outputs of an LLM from numbers back into text, we also need a way to turn token IDs into text.
# For this, we can create an inverse version of the vocabulary that maps token IDs back to corresponding text tokens.

# Let's implement a complete tokenizer class in Python.
# The class will have an encode method that splits text into tokens and carries out the string-to-integer mapping to produce token IDs via the vocabulary.
# In addition, we implement a decode method that carries out the reverse integer-to-string mapping to convert the token IDs back into text.

# Step 1: Store the vocabulary as a class attribute for access in the encode and decode methods
# Step 2: Create an inverse vocabulary that maps token IDs back to the original text tokens
# Step 3: Process input text into token IDs
# Step 4: Convert token IDs back into text
# Step 5: Replace spaces before the specified punctuation


In [20]:
class SimpleTokenizerV1:

    # __init__ method is called by default when a instance of SimpleTokenizerV1 is created
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # print(preprocessed)
        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations.lets suppose the joined tokens came out to be "Fox is chased ." So the problem is that
        # there is a space between . and chased. So below operation will output "Fox is chased."
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [21]:
# Let's instantiate a new tokenizer object from the SimpleTokenizerV1 class and tokenize a passage from Edith Wharton's short story to try 
# it out.

In [22]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [23]:
# The code above prints the following token IDs:
# Next, let's see if we can turn these token IDs back into text using the decode method:

In [24]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [25]:
# Based on the output above, we can see that the decode method successfully converted the token IDs back into the original text.

In [26]:
# Let's now apply it to a new text sample that is not contained in the training set. Meaning what if some words are not present in vocabulary.

In [27]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))
# The problem is that the word "Hello" was not used in the The Verdict short story.Hence, it is not contained in the vocabulary. 
# This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs.

KeyError: 'Hello'

In [ ]:
# The problem is that the word "Hello" was not used in the The Verdict short story.
# Hence, it is not contained in the vocabulary.
# This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs.

In [ ]:
# ADDING SPECIAL CONTEXT TOKENS
# In the previous section, we implemented a simple tokenizer and applied it to a passage from the training set.
# In this section, we will modify this tokenizer to handle unknown words.
# In particular, we will modify the vocabulary and tokenizer we implemented in the previous section, SimpleTokenizerV2, to support two new tokens, <|unk|> 
# and <|endoftext|>

In [ ]:
# We can modify the tokenizer to use an <|unk|> token if it encounters a word that is not part of the vocabulary.

# Furthermore, we add a token between unrelated texts.For example, when training GPT-like LLMs on multiple independent documents or books, 
# it is common to insert a token before each document or book that follows a previous text source

In [ ]:
# Let's now modify the vocabulary to include these two special tokens, <|unk|> and <|endoftext|>, by adding these to the list of all unique words that we 
# created in the previous section:

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"]) # <|endoftext|> and <|unk|> will be the last two tokens in the vocabulary
vocab = {token:integer for integer,token in enumerate(all_tokens)}

# When we are working with multiple text sources, we add <|endoftext|> tokens between the texts. Like if one text source comes from a book1 and 
# another text source from a news article. So the <|endoftext|> token act as markers, signalling the start and end of a particular segment. This 
# leads to more effective processing and understanding by the llm.

# Infact when GPT was trained the <|endoftext|> token was used between different text sources

In [ ]:
len(vocab.items())

In [ ]:
# Based on the output of the print statement above, the new vocabulary size is 1132 (the vocabulary size in the previous section was 1130).

In [ ]:
# As an additional quick check, let's print the last 5 entries of the updated vocabulary.

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

In [ ]:
# A simple text tokenizer that handles unknown words

# Step 1: Replace unknown words by <|unk|> tokens
# Step 2: Replace spaces before the specified punctuations

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text
        

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

In [ ]:
# Based on comparing the de-tokenized text above with the original input text, we know that the training dataset, Edith Wharton's short story The Verdict,
# did not contain the words "Hello" and "palace."

In [ ]:
# So far, we have discussed tokenization as an essential step in processing text as input to LLMs. Depending on the LLM, some researchers also 
# consider additional special tokens such as the following:

# [BOS] (beginning of sequence): This token marks the start of a text. It signifies to the LLM where a piece of content begins.

# [EOS] (end of sequence): This token is positioned at the end of a text, and is especially useful when concatenating multiple unrelated texts, 
# similar to <|endoftext|>. For instance, when combining two different Wikipedia articles or books, the [EOS] token indicates where one article ends 
# and the next one begins.

# [PAD] (padding): When training LLMs with batch sizes larger than one, the batch might contain texts of varying lengths. To ensure all texts have the 
# same length, the shorter texts are extended or "padded" using the [PAD] token, up to the length of the longest text in the batch.

In [ ]:
# Note that the tokenizer used for GPT models does not need any of these tokens mentioned above but only uses an <|endoftext|> token for simplicity.
# So GPT3, GPT4 when they are trained they don't use the unknown token, they only use the endoftext for simplicity.

In [ ]:
# the tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding 
# tokenizer, which breaks down words into subword units

# BYTE PAIR ENCODING

## This was used to train LLMs such as GPT-2, GPT-3 and the original model used in ChatGPT.

<img src="images/tokenization_algo.png" width="800">

# Types of tokenization algoritms:

## Word based tokenization 
## Sub-word based tokenization
## Character based tokenization

## In Word based tokenizer usually every word in the sentence is usually 1 token. The tokenizer above is the word based tokenizer. 
## The first problem is out of vocabulary(OOV) word. It is usually very difficult to deal with OOV words. Vocabulary is just a dictionary with 
## a collection with a collection of tokens arranged in ascending order and every token corroesponds to a token id.

## Another problem with this word based tokenization scheme is that suppose [boy ,boys] are 2 words in the vocabulary. Each of them will have different tokens.
## token ids can be very far apart or similar also but the probelem is that these words are very similar and when we do this tokenization this similarity is 
## not captured.

# Character based tokenization:

## Instead of having individual words as token. In this kind of character based tokenization individual characters are considered as tokens.
## this will lead to a very small vocabulary because very language has a fixed number of characters.
## So benefits are very small vocabulary and no OOV problem.

## The first problem is that meaning associated with words is completely lost. Essentially the advantage of dealing with languge models is that words 
## have meaning so different sentences and different words might be related to each other. Like boy and boys have a common meaning. This is completely
## lost since you are breaking it down into individual characters.

## Second problem is that tokenized sequence is much longer than the initial raw text.
## Both of these words have this disadvantage that the meaning between words is not captured. Boy and boys have a common root, that root is not captured

# Sub-word based tokenization

## Byte pair encoding is an example of sub-word based tokenization.

## Rule 1: When you get the dataset you don't split frequently used words into smaller subwords.
## Rule 2: Split the rare words meaning which are not occurring too many times, then you split these words into smaller meaningful subwords.

## So this is a mix between word tokenizer and character tokenizer.

## example if the word "boy" is appearing multiple times in the dataset then it should not be split further so "boy" is retained as token. But if the words "boys"
## is encountered that should be split into "boy" and "s". Because "boys" might not be appearing too many times and again "boys" also derives from the word "boy".
## So we should divide the word "boys" into smaller meaningful subwords.

# Benfit of Sub-word based tokenization
## The subword splitting helps the model learn that different words with the same root word such as for example token, tokens and tokenizing are similar in meaning.
## It also helps the model learn that "tokenization" and "modernization" are made up of different root words but have the same suffix "ization" and are used in 
## same syntactic situation.

## BPE(Byte pair encoding) is a subword tokenizaiton alogorithm

<img src="images/bpe_origin.png" width="800">

In [ ]:
## BPE algorithm was introduced in 1994. It is basically a data compression algorithm. We scan the data and most common pair of consecutive bytes
## of data is replaced with a byte that doesn't occur in data. So we identify pair of consecutive bytes which occure the most, so we find these pairs
## which occur the most frequently and then we replace them with a byte which does not exist in the data and we keep on doing this iteratively.

In [ ]:
## "aaabdaaabac"

## The byte pair "aa" occurs the most and we will replace it with "Z" as z doesn't occur in the data.

### compressed data : "ZabdZabac"

## "aaa" also occurs the most but "aaa" is not a byte pair it is of size 3.
## the next common byte pair is "ab". We will replace this by "Y".

## compressed data: "ZYdZYac"
## only byte pair left is ac. "ac" only appears once so we don't need to encode it.
## This process to replacing common byte pair with another variable is called encoding. Thats where the word encoder comes from.
## We can go 1 more layer deeper by further replacing zy with another variable which is lets say "w".
## SO the algorithm itself is very simple, you scan the data from left to right, identify the byte pairs which occur the most and then you replace them with
## a byte which doesn't exist in the data.

## You do this iteratively until you reach a stage where no byte pairs occurs more than once. This is the simple byte pair encoding algorithm.

In [ ]:
## BPE algorithm is slightly tweaked to convert entire sentence into subwords which will be very useful.

## BPE algorithm ensures that most common words in the vocabulary are represented as a single token, while rare words are broken down into two or more
## subword tokens.

# How is BPE alogorithm used for LLMs?

### Lets us consider below dataset of words:
### {"old": 7, "older": 3, "finest": 9, "lowest":4}

### for subword based tokenization using BPE algorithm:
### Preprocessing: We need to add end token end token "</w>" at the end of each word.
### {"old</w>": 7, "older</w>": 3, "finest</w>": 9, "lowest</w>":4}
### If we use the word based tokenisation there is no meaning that is captured. old is the root that is not captured and est is the common root that is not
### captured. So word based tokenization and character based tokenization they don't capture these meanings or root words.

## old appears 7 times in the dataset, older appears 3 times, finest appears 9 times and lowest appears 4 times.

## Let us now split the words into characters and count their frequency.

<img src="images/char_frequency_1.png" width="400">

## Next step in BPE algorithm is to look for the most frequent pairing. We will merge them and perform the sam iteration again and again until we reach the token limit or iteration limit

## Iteration 1: Start with the second most common token "e" as most probably the most common byte pair will be having "e". "es" pairing is appearing the most.

<img src="images/char_frequency_2.png" width="400">

## Iteration 2: "es" and "t" are byte pair appearing multiple times.

<img src="images/char_frequency_3.png" width="400">

## The previous word level tokenizers and the character level tokenizers could not identify that "est" is the common root multiple words, but with BPE alogorithm we have already created a token for "est".

## We can see that "est" and  "</w>" byte pair is appearing 13 times.

<img src="images/char_frequency_4.png" width="400">

## We could have left "est" but we combined "est" and "</w>" but then there would have been no difference between words like estimate and highest. So to encode
## the information that "est" is ending sequence we combined "est" and "</w>". So now the algorithm and the tokenizer can differentiate estimate and highest.

## "o" and "l" byte pair occurred 10 times.

<img src="images/char_frequency_5.png" width="400">

## "ol" and "d" byte pair appeared 10 times.

<img src="images/char_frequency_6.png" width="400">

## As you can see our BPE algorithm has captured meaning that two tokens "old" and "est</w>" are captured and these are subwords. They are neither characters nor words but they encode the root representation.

## "f", "i" and "n" appear nine times but we just have 1 word with these characters. So we are not merging them.


## Let us remove tokens with frequency of 0.

<img src="images/char_frequency_7.png" width="400">

## This is how the subword tokenizer works and this is exactly how BPE algorithm which is a subword tokenizer works for training models like GPT2 and GPT3.


## The stopping criteria for this iteration can be either token cout or number of iterations.

In [ ]:
## Since implementing BPE can be relatively complicated so will use an existing python open source library called tiktoken

# BYTE PAIR ENCODING

In [ ]:
! pip3 install tiktoken

In [ ]:
import importlib
import tiktoken

print("tiktoken version", importlib.metadata.version("tiktoken"))

In [ ]:
## once initialized we can instantiate BPE tokenizer from tiktoken:

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
## The usage of this tokenizer is similar to SimpleTokenizerV2

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
## someunknownPlace would have resulted in OOV problem. Advantage of BPE is that it has character and subwords both as token.

## 50256 is the token id of <|endoftext|> and this is also the vocabulary size of the tokenization scheme used in gpt2 or gpt3
## This english language is estimated to have 170000 and 200000 words in current usage. So if we would have used word based tokenizer then
## then vocabulary size would have been this much but with BPE algorithm tokenizer, the vocabulary size has been reduced by 1/3rd
## GPT3 and GPT4 has much higher level of tokens but even GPT2 has good enough performance.

## someunknownPlace was not encodeded then also we didn't got eny error. The tokenizer is able to encode random words like someunknownPlace.
## someunknownPlace is broken down into subwords it seems(617, 34680, 27271, 13).
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

In [ ]:
## <|endoftext|> is assigned a relatively large token id namley 50256 and that is the last tokenin BPE tokenizer used for GPT-2.

## Infact the BPE tokenizer which was used to train models such as GPT-2, GPT-3 and the original model used in chatgpt, has a total
## vocabulary size of 50257.

In [ ]:
## BPE tokenizer above encodes and decodes unknown words. The BPE tokenizer can handle any unknown word.

## The algorithm underlying BPE breaks down words that aren't defined in its predefined vocabulary into smaller subword units or even
## individual characters. This enables it to handle OOV words. So thanks toi BPE algorithm ,if the tokenizer encounters an unfamilar word
## during tokenization, it can represent it as a sequence of subword tokens or characters.

In [ ]:
## Another example to illustrate how the BPE tokenizer deals with unknown tokens.

In [ ]:
integers = tokenizer.encode("Akwirw ier") # completely random words
print(integers) # no error and even encoding succeeds
# tokenizer must have broken this string into words and subwords based on what is present in the vocabulary and then to each of these subword or character
# it would have assigned a token id.

strings = tokenizer.decode(integers)
print(strings)

### CREATING INPUT-TARGET PAIRS

<div class="alert alert-block alert-success">
In this section we implement a data loader that fetches the input-target pairs using a sliding window approach.</div>

<div class="alert alert-block alert-success">
To get started, we will first tokenize the whole The Verdict short story we worked with
earlier using the BPE tokenizer introduced in the previous section:</div>

<img src="images/input_output_pair.png" width="400">

### Block marked but blue will be the input to the llm and the block marked with red will be the target or output which the llms have to learn.

### All rows are interations.

### First iteration the input is "llm" and based on this input the llm has to learn the output which is "learn".

### Whatever comes after the prediction is masked or not shown to the llm, this is what happens in the iteration number 1.

### In iteration number 2 "learn" which was output or target in the first iteration now is a part of the input, so in the 2nd iteration
### "llms learn" that is the input and "to" is the target.

### There are two things to notice here. Within the sentence itself we break down the sentence itself into input and target which is the next word.
### Second thing to remember is that in subsequent iterations whatever was the output in the previous iteration then becomes the input. So this is 
### a auto regressive model and it is also called the self supervised learning or you can think of it as unsupervised learning itself because we 
### are not labelling the input or the output, the sentence structure itself is used to predict or is used to determine what is input and the output.

### in cats and dog image classifier we have to manually label.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

<div class="alert alert-block alert-info">
    
Executing the code above will return 5145, the total number of tokens in the training set,
after applying the BPE tokenizer.

</div>

<div class="alert alert-block alert-success">
Vocabulary is essentially a dictionary which maps the tokens into token ids. Since we are using the BPE the tokens won't be words
but they can characters or subwords also. The size 5145 conveys is that our vocabulary for this text which we have as a dataset has a length
of 5145 which means we have 5145 tokens and corroesponding token id</div>

<div class="alert alert-block alert-success">
Next, we remove the first 50 tokens from the dataset for demonstration purposes as it
results in a slightly more interesting text passage in the next steps:</div>

In [ ]:
enc_sample = enc_text[50:]

<div class="alert alert-block alert-success">
One of the easiest and most intuitive ways to create the input-target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input tokens
and y contains the targets, which are the inputs shifted by 1:</div>

<div class="alert alert-block alert-info">
The context size determines how many tokens are included in the input

</div>

In [ ]:
context_size = 4 # length of the input
# The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens) 
# to predict the next word in the sequence. 
# The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

# Intitutively context_size is basically how many words the model should pay attention at one time to predict the next word.

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")


<div class="alert alert-block alert-success">
Processing the inputs along with the targets, which are the inputs shifted by one position,
we can then create the next-word prediction tasks as
follows:</div>

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    
    print(context, " ------> ", desired)

# There are 4 prediction tasks here. When i created this input output pair of X and Y, it is not 1 prediction task it is actually 4 prediction tasks
# which are happening here. If context size would have been 8 then there would have been 8 prediction tasks in each input output pair.

# usually in regression and classifcation problem 1 input output pair corroesponds to 1 prediction tasks. Like image of a dog needs to be classified as
# dog or cat. But in case of llms one input output pair corroesponds to the number of prediction tasks as set by the context size.

<div class="alert alert-block alert-info">
Everything left of the arrow (---->) refers to the input an LLM would receive, and the token
ID on the right side of the arrow represents the target token ID that the LLM is supposed to
predict.
</div>

<div class="alert alert-block alert-success">
For illustration purposes, let's repeat the previous code but convert the token IDs into
text:</div>

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), " ---> ", tokenizer.decode([desired]))

<div class="alert alert-block alert-warning">

We've now created the input-target pairs that we can turn into use for the LLM training in
upcoming chapters.
    
</div>

<div class="alert alert-block alert-warning">

There's only one more task before we can turn the tokens into embeddings:implementing an efficient data loader that
iterates over the input dataset and returns the inputs and targets as PyTorch tensors, which
can be thought of as multidimensional arrays.
    
</div>

<div class="alert alert-block alert-warning">

In particular, we are interested in returning two tensors: an input tensor containing the
text that the LLM sees and a target tensor that includes the targets for the LLM to predict,
    
</div>

### IMPLEMENTING A DATA LOADER

<div class="alert alert-block alert-success">
For the efficient data loader implementation, we will use PyTorch's built-in Dataset and
DataLoader classes. Dataset and dataloader enables you to load or process the data in a much more efficent and compact manner</div>

<div class="alert alert-block alert-info">
    
Step 1: Tokenize the entire text
    
Step 2: Use a sliding window to chunk the book into overlapping sequences of max_length

Step 3: Return the total number of rows in the dataset

Step 4: Return a single row from the dataset
</div>

In [ ]:
# Dataset is: just a class defined by a library (PyTorch)

# In Python terms: Dataset = a normal class that follows a protocol

# That protocol is:
# __len__()
# __getitem__()

# PyTorch’s Dataset is basically:

# class Dataset:
#     def __getitem__(self, index):
#         raise NotImplementedError

#     def __len__(self):
#         raise NotImplementedError

from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))


    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

<div class="alert alert-block alert-warning">

The GPTDatasetV1 class is based on the PyTorch Dataset class.

It defines how individual rows are fetched from the dataset. 

Each row consists of a number of
token IDs (based on a max_length) assigned to an input_chunk tensor. 

The target_chunk
tensor contains the corresponding targets. 

I recommend reading on to see how the data
returned from this dataset looks like when we combine the dataset with a PyTorch
DataLoader -- this will bring additional intuition and clarity.
    
</div>

<div class="alert alert-block alert-success">
The following code will use the GPTDatasetV1 to load the inputs in batches via a PyTorch
DataLoader:</div>

<div class="alert alert-block alert-info">
    
Step 1: Initialize the tokenizer

Step 2: Create dataset

Step 3: drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes
during training

Step 4: The number of CPU processes to use for preprocessing
    
</div>

In [ ]:
# txt takes the text file which is the data set we have.

# max_length is the context size. Earlier for demonstration we tried 4. But when gpt2 or gpt3 those high level llm models are designed
# they ususally implement the context length of 256 which means they are so strong that the model can look at 256 words and predict the 
# next word during training.

# stride is 128 means when we create input output batches how much we need to skip before we create the next batch

# num_workers is also the number of CPU threads which we can run simultaneously.

# batch_size is basically number of batches the model processes at once before updating its parameters. So to make sure that model updates its 
# parameters quickly, the data is usually chunked into batches so that after analysing 4 batches in this case the model will update its parameters
# rather than going through entire dataset.

# num_workers is for parallel processing on different threads of the CPU.
def create_dataloader_v1(txt, batch_size = 4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    
    # Initialize the tokenizer. tiktoken is the BPE used by GPT
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

<div class="alert alert-block alert-success">
    
Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4, 

This will develop an intuition of how the GPTDatasetV1 class and the
create_dataloader_v1 function work together: </div>

<img src="images/stride_1_vs_stride_4.png" width="400">

# More stride means that you move over the data in faster manner and lesser computations will be there.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

<div class="alert alert-block alert-info">
    
Convert dataloader into a Python iterator to fetch the next entry via Python's built-in next() function
    
</div>

In [ ]:
import torch
print("pytorch version: ", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1,max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

<div class="alert alert-block alert-warning">

The first_batch variable contains two tensors: the first tensor stores the input token IDs,
and the second tensor stores the target token IDs. 

Since the max_length is set to 4, each of the two tensors contains 4 token IDs. 

Note that an input size of 4 is relatively small and only chosen for illustration purposes. It is common to train LLMs with input sizes of at least
256.
    
</div>

<div class="alert alert-block alert-success">
    
To illustrate the meaning of stride=1, let's fetch another batch from this dataset: </div>

In [ ]:
second_batch = next(data_iter)
print(second_batch)

<div class="alert alert-block alert-warning">

If we compare the first with the second batch, we can see that the second batch's token
IDs are shifted by one position compared to the first batch. 

For example, the second ID in
the first batch's input is 367, which is the first ID of the second batch's input. 

The stride
setting dictates the number of positions the inputs shift across batches, emulating a sliding
window approach
    
</div>

<div class="alert alert-block alert-warning">

Batch sizes of 1, such as we have sampled from the data loader so far, are useful for
illustration purposes. 
                                                                                 
If you have previous experience with deep learning, you may know
that small batch sizes require less memory during training but lead to more noisy model
updates.

Just like in regular deep learning, the batch size is a trade-off and hyperparameter
to experiment with when training LLMs.
    
</div>

<div class="alert alert-block alert-success">
    
Before we move on to the two final sections of this chapter that are focused on creating
the embedding vectors from the token IDs, let's have a brief look at how we can use the
data loader to sample with a batch size greater than 1: </div>

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

# Here if look at first input and second then there is no overlap because stride is equal to 4.
# And here batch_size is equal to 8 so see the input tensor has essentially 8 inputs and the output tensor will also have 8 rows because
# the batch_size is equal to 8. So the model will have essentially process this batch before making the parameter updates. 

<div class="alert alert-block alert-info">
    
Note that we increase the stride to 4. This is to utilize the data set fully (we don't skip a
single word) but also avoid any overlap between the batches, since more overlap could lead
to increased overfitting.
    
</div>

### CREATING TOKEN EMBEDDINGS

In [ ]:
### Representing words numerically

### We have words and we want these words to be inputs to machine learning model or large language model. But computers can't understand words.
### So we need to represent words in the format of numbers.

### computers need numerical representation of words

### Can we assign random numbers to each words. We already converted the tokens into token ids and we have manitained a vocabulary, these token 
### ids can be training input to the llm

### We can not just use randomly assigned numbers and the main problem is that the beauty of language  is that some words are related to other words for
### example cat and kitten are related but just assigning random numbers or token ids to each word doesnot really capture the semantic meaning between these 
### individual words

<img src="images/cat.png" width="400">
### Why convolution neural networks so well: Because CNN don't just use the pixel values and stretch it out as 1 input vector. They actually encode the 
### the spatial relation between the pixels.

### Two eyes are close ot each other, whiskers are closer to eyes, two ears are close to each other, that is an information that is contained in the image
### itself and we should exploit this information when we feed the input to the model. If we don't exploit the information which is inherently present in
### the image we are not doing an optimal thing. Thats why CNN work so well, they exploit the spatial relation between pixels.

In [ ]:
### When we look at sentences we as humans are able to understand what sentences mean because words carry meanings and there are some  words which are closer
### in meaning to other words.

### This is inherent advantage in the text which we need to exlpoit. We need to exploit the fact that cat and kitten are closer to each other. If we don't 
### exploit this information we will train for a huge amount of time and we won't be doing an optimal machine learning training.

<img src="images/one_hot_encoding_1.png" width="400">
<img src="images/one_hot_encoding_1.png" width="400">


In [ ]:
### I will have a huge vocabulary of all possible words and then I will assgin one hot encoding. So to every word there will be all zeroes but there
### will be one 1. But this will result to a similar problem with random number assignment. One hot encoding also fails to capture the semantic relationship
### between words.

<img src="images/semantic_similar_words.png" width="400">


In [ ]:
### Then how do you encode the semantic relationship or semantic meaning when you are going to convert these words to numbers.

### Then came the idea that what if every word was encoded as a vector.
### We might question what should be the dimension of the vector(2D, 3D, thousand dimension vector).

### What if the dimension is determined by features, In the above image we have mentioned 5 features.
### Based on these feature/questions answer if yes very high number

### Based on the vector above we can see that they are closer to each other because whichever value is high for dog is high for cat also. and whichever is low 
### for dog is low for cat also.
### If we look at banana and apple they look closer to each other.

### So if represent words as vectors and if we construct these vectors in a smart manner, vectors can capture semantic meaning which was not captured before
### when we did one hot encoding and random number assignment. 

### These are called vector embeddings or token embeddings because every token is converted to a vector embedding.

In [ ]:
### How do you construct these embeddings? How do i make these vectors so that lets says dog and puppy are closer but dog and banana are farther apart.

### We have to train a neural network to create vector embedding.
### So for example we have all the imformations all the tokens and we have some output and based on this information and output we have to train a neural
### network to make sure that the vector embeddings is correct. This information comes from text. So based on sentences in a textual document we know which
### words are closer to each other, which words are similar to each other, those should have similar vectors thats the training data and we train a neural network
### to construct a vector embedding.

### Creating this vector embedding is not easy, imagive there is a vocabulary of 50000 thousans words and we have to create vector embedding for these many
### words. This is computationally expensive and thats why training GPT takes a huge amount of time.

In [ ]:
### Many big companies like google have pre-trained token embeddings
### word2vec: There is a google news dataset which has about 100 billion words. So this word2vec-google-news-300 are already pre-trained vectors on this
### huge dataset. So google has already trained the neural network to create this vector embeddings.
### We get this google news dataset 100 billion words and then we do the training to map every word to a vector. 300 is basically the number of dimensions.
### When we create token embeddings or vector embeddings, words are mapped to a large dimensional vector space.
### So here every word is transformed to a 300 dimesional vector and then we train based on the underlying data so that the semantic meaning between the
### words is preserved.

### This is already a pre-trained dataset
### When GPT was built or a large language models are built, they don't use a pre-trained dataset, they train these embeddings along with training a large 
### language model itself.

### go through Copy of Vector embedding.ipynb in google drive to see the word2vec-300 in action.

<img src="images/embedding_weight_matrix.png" width="400"/>

In [ ]:
### How are token embeddings created for LLM?

### We start with vocabulary for LLM and then we have tokens in the vocabulary and we have tokens in the vocabulary and then we have token ids.

### Every token id is converted to embedding vectors.
### The above output iamge is called embedding weight matrix.

### Two things are needed before we can construct this embedding weight matrix. Vocabulary size and vector dimension.

### Vector embedding dimension for GPT-2 was 768 for smallest model.
### vocabulary size for GPT-2 was 50257 which means GPT-2 had these many tokens. Those were subwords made through BPE.

### SO token ids went from 0,1, , , ,50257 and then for each of these token ids there would be a vector and the vector dimension was 768 in this case.
### So for 0th token id there would be a vector of 768 dimensions and same for all the token ids.
### So size of this embedding layer weight matrix would be 50257*768.

### For 1st token id there would be 768 weights and for seconf token id also the same.

### Once we get the token id, you convert these into embedding layer weight matrix.

In [ ]:
### Intially when this weight matrix is initiallized we don't know how the vectors are. word2vec is a pre-trained model.

### We just now know the size of the matrix. But we don't know what each weight will be.
### So we initialize the embedding weight with random values. So all of these 50257*768 values are initialized with randomly.
### This initliazation serves as the starting point for the LLM learning process.
### These embedding weights are optimized as part of the LLM training process.

### When GPT-2 was trained these values were not known before what were the ideal weight values.
### A training process was implemented where we had these many parameters 50257*768 parameters and each of these weight parameters were optimized
### during the training process. And that is how vector embeddings or token embeddings were created.
### How was the optmization process done?: We had the training data and based on the training data we knew which words were closer to each other and
### which words were farther apart from each other.
### If we are aware of neural network training, back propagation is implemented here to optimise all of these weights of this embedding layer weight matrix.

### So if we think about this at the heart of the LLM are just giant neural networks and one part of this giant neural network is training an embedded layer 
### matrix

### Token embeddings are fed as input to the training, but during the training of this GPT, while this its training to predict the next word we also train 
### that embedding neual network itself.

<div class="alert alert-block alert-success">
    
Let's illustrate how the token ID to embedding vector conversion works with a hands-on
example. Suppose we have the following four input tokens with IDs 2, 3, 5, and 1:</div>

In [ ]:
input_ids = torch.tensor([2,3,5,1])

<div class="alert alert-block alert-success">
    
For the sake of simplicity and illustration purposes, suppose we have a small vocabulary of
only 6 words (instead of the 50,257 words in the BPE tokenizer vocabulary), and we want
to create embeddings of size 3 (in GPT-3, the embedding size is 12,288 dimensions):

</div>

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
# Embedding is simple lookup table in pytorch that stores embeddings of a fixed dictionary and size.
# This initializes the weights of the embedding matrix in random manner.
print(embedding_layer)

<div class="alert alert-block alert-info">
    
The print statement in the code prints the embedding layer's underlying
weight matrix:
    
</div>

In [ ]:
print(embedding_layer.weight)

<div class="alert alert-block alert-info">
    
We can see that the weight matrix of the embedding layer contains small, random values.
These values are optimized during LLM training as part of the LLM optimization itself, as we
will see in upcoming chapters. Moreover, we can see that the weight matrix has six rows
and three columns. There is one row for each of the six possible tokens in the vocabulary.
And there is one column for each of the three embedding dimensions.
    
</div>

<div class="alert alert-block alert-success">
    
After we instantiated the embedding layer, let's now apply it to a token ID to obtain the
embedding vector:

</div>

In [ ]:
print(embedding_layer(torch.tensor([3])))

<div class="alert alert-block alert-info">
    
If we compare the embedding vector for token ID 3 to the previous embedding matrix, we
see that it is identical to the 4th row (Python starts with a zero index, so it's the row
corresponding to index 3). In other words, the embedding layer is essentially a look-up
operation that retrieves rows from the embedding layer's weight matrix via a token ID.
    
</div>

<div class="alert alert-block alert-success">
    
Previously, we have seen how to convert a single token ID into a three-dimensional
embedding vector. Let's now apply that to all four input IDs we defined earlier
(torch.tensor([2, 3, 5, 1])):

</div>

In [ ]:
print(embedding_layer(input_ids))

<div class="alert alert-block alert-info">
    
Each row in this output matrix is obtained via a lookup operation from the embedding
weight matrix
    
</div>

<img src="images/neural_network_linear_layer.png" width="400"/>

In [ ]:
### This embedding layer is same as neural network linear layer.
### Embedding matrix under the hood is doing the same operation as a neural network linear layer. So what an embedding does is that you have inputs,
### those are converted into 1 hot representation and they are fed into the neural network with 5 neurons(vector dimension is 5). The output of linear layer
### is X*W. This gives the same output as embedding_layer.

### Earlier we saw torch.nn.Embedding and here we can see torch.nn.Linear, then you might think why linear is not used to define the embedding matrix.
### Beacuse both will lead to same output but embedding_layer is much more computationally efficient because in linear we have mulitple unncessary multiplications
### with 0.